# 04_resnet34.ipynb - ResNet-34 Training

**Project**: BSM Project 3 - D-ResNet Paper Reproduction
**Model**: ResNet-34 (pretrained on ImageNet)
**Paper Target**: Accuracy=0.83, Precision=0.89, Recall=0.83
**Step**: 4 of 9 - Model Training

## Configuration
- Input: 224Ã—224Ã—3
- Backbone: ResNet-34 (ImageNet pretrained)
- Head: Linear(512, 6) + Softmax
- Loss: CrossEntropyLoss
- Optimizer: Adam (lr=1e-4, weight_decay=1e-4)
- Batch size: 32
- Epochs: 60
- Seed: 42
- Data: Primary split (combined dataset, 70/15/15)
- Checkpoint: Best validation accuracy

In [ ]:
# ========== CONFIGURATION ==========
SEED = 42
EPOCHS = 1  # CHANGE TO 60 FOR FULL TRAINING AFTER SMOKE TEST PASSES
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 6
IMG_SIZE = 224
DEVICE = "cuda"

PRIMARY_SPLIT_PATH = "/kaggle/working/data_info/primary_split.json"PRIMARY_SPLIT_PATH = "/kaggle/working/data_info/primary_split.json"
OUTPUT_DIR = "/kaggle/working/models/resnet34"
RESULTS_DIR = "/kaggle/working/results"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "confusion_matrices"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "metrics"), exist_ok=True)

print(f"EPOCHS = {EPOCHS} (set to 60 for full training)")
print(f"BATCH_SIZE = {BATCH_SIZE}")
print(f"LR = {LR}")

In [ ]:
# ========== IMPORTS & SEED ==========
import os
import sys
import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from PIL import Image
import cv2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Seed: {SEED}")

In [ ]:
# ========== LOAD SPLIT DATA ==========
with open(PRIMARY_SPLIT_PATH, 'r') as f:
    split_data = json.load(f)

train_data = split_data["train"]
val_data = split_data["val"]
test_data = split_data["test"]

print(f"Train samples: {len(train_data)}")
print(f"Val samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

CLASS_NAMES = sorted(set(item["class"] for item in train_data))
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: cls for cls, i in CLASS_TO_IDX.items()}

print(f"Classes: {CLASS_NAMES}")
print(f"Class to idx: {CLASS_TO_IDX}")

In [ ]:
# ========== DATASET CLASS ==========
class BSMDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        item = self.data_list[idx]
        img_path = item["path"]
        label = CLASS_TO_IDX[item["class"]]
        
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Failed to load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            if hasattr(self.transform, 'augment'):
                augmented = self.transform(image=image)
                image = augmented["image"]
            else:
                image = self.transform(image)
        
        return image, label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = BSMDataset(train_data, transform=train_transform)
val_dataset = BSMDataset(val_data, transform=val_test_transform)
test_dataset = BSMDataset(test_data, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# ========== MODEL: ResNet-34 ==========
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

num_ftrs = model.fc.in_features  # 512 for ResNet-34
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

print(f"Model: ResNet-34")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ========== TRAINING LOOP ==========
best_val_acc = 0.0
best_epoch = 0
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    scheduler.step(val_acc)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
        }, os.path.join(OUTPUT_DIR, "best_model.pth"))
    
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.1f}s | Best Val Acc: {best_val_acc:.4f} (epoch {best_epoch+1})")

total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.1f} minutes")
print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch+1}")

In [ ]:
# ========== PLOT TRAINING CURVES ==========
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], 'b-', label='Train Loss')
axes[0].plot(epochs_range, history["val_loss"], 'r-', label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('ResNet-34: Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], 'b-', label='Train Acc')
axes[1].plot(epochs_range, history["val_acc"], 'r-', label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('ResNet-34: Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'resnet34_training_curves.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(RESULTS_DIR, 'resnet34_training_curves.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ========== LOAD BEST MODEL FOR TEST EVALUATION ==========
checkpoint = torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from epoch {checkpoint['epoch']+1} with val_acc={checkpoint['val_acc']:.4f}")

model.eval()

In [ ]:
# ========== TEST SET EVALUATION ==========
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {test_acc:.4f}")

precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
recall_per_class = recall_score(all_labels, all_preds, average=None, zero_division=0)
f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

precision_weighted = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall_weighted = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"\nPer-class Metrics:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls}: Precision={precision_per_class[i]:.4f}, Recall={recall_per_class[i]:.4f}, F1={f1_per_class[i]:.4f}")

print(f"\nWeighted Metrics:")
print(f"  Precision: {precision_weighted:.4f}")
print(f"  Recall:    {recall_weighted:.4f}")
print(f"  F1:        {f1_weighted:.4f}")

In [ ]:
# ========== CONFUSION MATRIX ==========
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'ResNet-34 Confusion Matrix (Test Acc: {test_acc:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices', 'resnet34_cm.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices', 'resnet34_cm.pdf'), bbox_inches='tight')
plt.show()

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('ResNet-34 Normalized Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrices', 'resnet34_cm_normalized.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========== SPECIFICITY & RMSE ==========
specificities = []
for i in range(NUM_CLASSES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    specificities.append(spec)

print(f"Per-class Specificity:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls}: {specificities[i]:.4f}")

one_hot = np.eye(NUM_CLASSES)[all_labels]
rmse = np.sqrt(np.mean((one_hot - all_probs) ** 2))
print(f"\nRMSE (one-hot vs softmax): {rmse:.4f}")

rmse_per_class = []
for i in range(NUM_CLASSES):
    class_mask = (all_labels == i)
    if class_mask.sum() > 0:
        class_rmse = np.sqrt(np.mean((one_hot[class_mask] - all_probs[class_mask]) ** 2))
        rmse_per_class.append(class_rmse)
    else:
        rmse_per_class.append(0)

print(f"Per-class RMSE:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls}: {rmse_per_class[i]:.4f}")

In [ ]:
# ========== PER-CLASS TABLE ==========
results_table = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": precision_per_class,
    "Recall (Sensitivity)": recall_per_class,
    "Specificity": specificities,
    "F1-Score": f1_per_class,
    "RMSE": rmse_per_class,
    "Support": [cm[i, :].sum() for i in range(NUM_CLASSES)]
})

weighted_row = pd.DataFrame({
    "Class": ["Weighted Avg"],
    "Precision": [precision_weighted],
    "Recall (Sensitivity)": [recall_weighted],
    "Specificity": [np.mean(specificities)],
    "F1-Score": [f1_weighted],
    "RMSE": [rmse],
    "Support": [len(all_labels)]
})

results_table = pd.concat([results_table, weighted_row], ignore_index=True)

print(results_table.to_string(index=False))

results_table.to_csv(os.path.join(RESULTS_DIR, 'metrics', 'resnet34_metrics.csv'), index=False)
print(f"\nMetrics saved to {RESULTS_DIR}/metrics/resnet34_metrics.csv")

In [ ]:
# ========== PREDICTED VS ACTUAL IMAGE GRID ==========
n_samples = min(18, len(test_data))
indices = np.random.choice(len(test_data), n_samples, replace=False)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.flatten()

for idx, ax in zip(indices, axes):
    item = test_data[idx]
    img_path = item["path"]
    true_label = item["class"]
    pred_idx = all_preds[idx]
    pred_label = CLASS_NAMES[pred_idx]
    pred_prob = all_probs[idx, pred_idx]
    
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    ax.imshow(image)
    ax.axis('off')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({pred_prob:.2f})", 
                color=color, fontsize=9)

plt.suptitle('ResNet-34: Predicted vs Actual (Green=Correct, Red=Wrong)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'resnet34_predictions_grid.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(RESULTS_DIR, 'resnet34_predictions_grid.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ========== SAVE FINAL RESULTS ==========
final_results = {
    "model": "ResNet-34",
    "epochs_trained": EPOCHS,
    "best_epoch": best_epoch + 1,
    "best_val_acc": float(best_val_acc),
    "test_acc": float(test_acc),
    "precision_weighted": float(precision_weighted),
    "recall_weighted": float(recall_weighted),
    "f1_weighted": float(f1_weighted),
    "rmse": float(rmse),
    "per_class": {
        cls: {
            "precision": float(precision_per_class[i]),
            "recall": float(recall_per_class[i]),
            "specificity": float(specificities[i]),
            "f1": float(f1_per_class[i]),
            "rmse": float(rmse_per_class[i]),
            "support": int(cm[i, :].sum())
        }
        for i, cls in enumerate(CLASS_NAMES)
    },
    "confusion_matrix": cm.tolist(),
    "training_history": history,
    "config": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "Adam",
        "scheduler": "ReduceLROnPlateau",
        "img_size": IMG_SIZE,
        "pretrained": True
    }
}

with open(os.path.join(RESULTS_DIR, 'metrics', 'resnet34_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"Final results saved to {RESULTS_DIR}/metrics/resnet34_results.json")

print("="*60)
print("RESNET-34 SUMMARY")
print("="*60)
print(f"Paper Target: Acc=0.83, Prec=0.89, Rec=0.83")
print(f"Our Result:   Acc={test_acc:.4f}, Prec={precision_weighted:.4f}, Rec={recall_weighted:.4f}")
print(f"Difference:   Acc={test_acc-0.83:+.4f}, Prec={precision_weighted-0.89:+.4f}, Rec={recall_weighted-0.83:+.4f}")